# 03 - Feature Engineering

**Phases 2 and 3.** `properties.csv` is a transaction ledger; the clustering
unit is the *client*. This notebook collapses 7,305 sold transactions into one
behavioural profile per client and joins it to the demographic table.


In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from src import config as cfg
from src.data_cleaning import load_and_clean
from src.feature_engineering import (aggregate_properties,
                                     build_client_features,
                                     feature_dictionary)
clients, properties, _ = load_and_clean(save=False)


## Aggregate the ledger

In [2]:
behaviour = aggregate_properties(properties)
print(behaviour.shape)
behaviour.head()


(2000, 21)


,client_id,total_properties,total_investment,avg_property_price,max_property_price,min_property_price,price_dispersion,avg_floor_area,total_area,avg_price_per_sqft,unique_towers,first_purchase,last_purchase,active_months,apartment_count,office_count,apartment_share,office_share,purchase_span_days,tower_diversity,purchase_intensity
0,C0001,4,1246764.72,311691.18,496266.41,175599.9,143699.387931,983.885000,3935.54,319.963172,4,2024-10-01,2025-12-01,4,4,0,1.0,0.0,426,1.0,1.0
1,C0002,5,1841095.93,368219.186,505127.63,248525.12,106902.667058,1187.942000,5939.71,313.167909,5,2024-01-01,2025-12-01,5,5,0,1.0,0.0,700,1.0,1.0
2,C0003,5,1661457.59,332291.518,584611.99,216874.97,150963.061495,1058.110000,5290.55,310.797967,5,2024-07-01,2025-10-01,5,5,0,1.0,0.0,457,1.0,1.0
3,C0004,6,1608263.51,268043.918333,606634.75,149009.73,168321.92692,937.103333,5622.62,275.564086,5,2024-02-01,2025-08-01,5,6,0,1.0,0.0,547,0.833333,1.2
4,C0005,13,3653385.38,281029.644615,628812.33,189194.31,123629.325683,927.296154,12054.85,299.398863,5,2024-02-01,2025-05-01,12,13,0,1.0,0.0,455,0.384615,1.083333


In [3]:
# Mirrors run_pipeline.py exactly, including the label-encoded columns, so
# re-running this notebook reproduces the committed artefact rather than
# overwriting it with a narrower table.
from src.preprocessing import label_encode

features = build_client_features(clients, properties)
features, encoders = label_encode(features)
features.to_csv(cfg.CLIENT_FEATURES, index=False)

print("client feature table:", features.shape)
print("units aggregated  :", int(features['total_properties'].sum()), "(expect 7,305)")
print("capital aggregated: $%s" % format(features['total_investment'].sum(), ',.2f'))
print("label-encoded columns:", [c for c in features.columns if c.endswith('_code')])
feature_dictionary()


client feature table: (2000, 42)
units aggregated  : 7305 (expect 7,305)
capital aggregated: $2,520,750,960.84
label-encoded columns: ['country_code', 'region_code', 'referral_channel_code']


,group,feature,description
0,behavioural,total_properties,Number of units purchased (purchase frequency)
1,behavioural,total_investment,"Total capital deployed, USD"
2,behavioural,avg_property_price,"Mean ticket size per unit, USD"
3,behavioural,max_property_price,"Largest single purchase, USD"
4,behavioural,price_dispersion,Std. dev. of ticket size (0 for single buyers)
5,behavioural,avg_floor_area,"Mean unit size, sq ft"
6,behavioural,total_area,"Total floor area acquired, sq ft"
7,behavioural,avg_price_per_sqft,"Mean unit price intensity, USD/sq ft"
8,behavioural,unique_towers,Distinct towers bought into (diversification)
9,behavioural,tower_diversity,"unique_towers / total_properties, 0-1"


## Distributions and redundancy

In [4]:
cols = ['total_properties','total_investment','avg_property_price',
        'max_property_price','price_dispersion','avg_floor_area','total_area',
        'avg_price_per_sqft','unique_towers','tower_diversity','office_share',
        'purchase_span_days','active_months','purchase_intensity']
features[cols].describe().T.round(2)


,count,mean,std,min,25%,50%,75%,max
total_properties,2000.0,3.6525,0.8397,3.0,3.0,4.0,4.0,13.0
total_investment,2000.0,1260375.48042,347830.664827,463611.95,1025238.005,1220893.165,1441973.8825,3653385.38
avg_property_price,2000.0,347089.955869,69721.821814,154537.316667,295807.677708,341523.383333,390843.316875,563423.5
max_property_price,2000.0,480546.955875,94300.835147,193265.77,411678.305,488390.735,550353.9525,736652.27
price_dispersion,2000.0,122943.510674,48238.481692,1700.004448,88995.980657,123627.952296,157148.387662,292994.963824
avg_floor_area,2000.0,1147.479668,219.922587,564.01,984.9525,1129.176667,1296.5625,1800.45
total_area,2000.0,4168.04248,1130.165735,1692.03,3387.53,4061.095,4740.3275,12054.85
avg_price_per_sqft,2000.0,302.509374,15.969587,246.9219,292.160539,302.265272,313.03815,373.744858
unique_towers,2000.0,3.607,0.701282,3.0,3.0,4.0,4.0,8.0
tower_diversity,2000.0,0.992831,0.042242,0.384615,1.0,1.0,1.0,1.0


In [5]:
corr = features[cols].corr()
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(cols)
         for b in cols[i+1:] if abs(corr.loc[a, b]) >= 0.75]
print("Strongly correlated pairs (|r| >= 0.75):")
for a, b, r in sorted(pairs, key=lambda t: -abs(t[2])):
    print(f"  {a:20s} ~ {b:20s} r = {r:+.2f}")
print()
cv = (features[cols].std() / features[cols].mean()).abs().sort_values()
print("Coefficient of variation (near-zero => near-constant, useless):")
print(cv.round(4).to_string())


Strongly correlated pairs (|r| >= 0.75):
  total_investment     ~ total_area           r = +0.98
  avg_property_price   ~ avg_floor_area       r = +0.96
  total_properties     ~ unique_towers        r = +0.93
  total_properties     ~ active_months        r = +0.85
  unique_towers        ~ active_months        r = +0.79
  avg_property_price   ~ max_property_price   r = +0.75



Coefficient of variation (near-zero => near-constant, useless):
tower_diversity       0.0425
avg_price_per_sqft    0.0528
purchase_intensity    0.1708
avg_floor_area        0.1917
unique_towers         0.1944
max_property_price    0.1962
avg_property_price    0.2009
total_properties      0.2299
active_months         0.2354
total_area            0.2712
total_investment       0.276
price_dispersion      0.3924
purchase_span_days    0.4034
office_share          1.2424


Two consequences for the model:

* `tower_diversity` (CV 0.04), `avg_price_per_sqft` (CV 0.05) and
  `purchase_intensity` (CV 0.17) are close to constant and carry almost no
  discriminating signal.
* `total_area ~ total_investment` (r = 0.98) and
  `avg_floor_area ~ avg_property_price` (r = 0.96) are near-restatements,
  because price here is essentially a linear function of area.

Whether pruning the redundant pair *helps* is an empirical question, so
notebook 04 tests both a pruned and a full behavioural feature set rather than
assuming an answer.
